In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import random
import concurrent.futures
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score

# ==========================================
# 0. CONFIGURATION
# ==========================================
RUN_CONFIG = [
    (50, False, 50000, 128, 0.0, 0.0),
    (50, True,  50000, 128, 1.0, 1.0),
    (500, False, 50000, 128, 0.0, 0.0),
    (500, True,  50000, 128, 1.0, 1.0),
    (5000, False, 2000, 128, 0.0, 0.0),
    (5000, True,  2000, 128, 1.0, 1.0),
]

LR = 0.01
NUM_GPUS_AVAILABLE = torch.cuda.device_count() if torch.cuda.is_available() else 1

# ==========================================
# 1. NTK Architecture
# ==========================================
class NTKLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.bias = nn.Parameter(torch.randn(out_features))
        self.c = in_features ** 0.5 

    def forward(self, x):
        return F.linear(x, self.weight, self.bias) / self.c

class WideMLP(nn.Module):
    def __init__(self, input_dim, width, output_dim):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            NTKLinear(input_dim, width),
            nn.ReLU()
        )
        self.final_layer = NTKLinear(width, output_dim)

    def forward(self, x):
        feat = self.feature_extractor(x)
        return self.final_layer(feat), feat


# ==========================================
# 2. Data Setup
# ==========================================
def process_mnist(dataset, N):
    imgs = dataset.data[:N].unsqueeze(1).float() / 255.0
    imgs_pooled = F.adaptive_avg_pool2d(imgs, (10, 10))
    X = imgs_pooled.view(N, -1)
    Y = dataset.targets[:N].long()
    return X, Y

mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True)
mnist_test = torchvision.datasets.MNIST(root='./data', train=False, download=True)

X_train_all, Y_train_all = process_mnist(mnist_train, 1000)
X_test_all, Y_test_all = process_mnist(mnist_test, 500)

# ==========================================
# 3. STRONG MIA ATTACK (FIXED + AUC)
# ==========================================
def audit_privacy(model, X_mem, Y_mem, X_non, Y_non, device):
    from sklearn.metrics import roc_auc_score
    import numpy as np
    import torch
    import torch.nn.functional as F

    model.eval()

    with torch.no_grad():
        logit_mem, _ = model(X_mem.to(device))
        logit_non, _ = model(X_non.to(device))

        prob_mem = F.softmax(logit_mem, dim=1)
        prob_non = F.softmax(logit_non, dim=1)

        # ========= features =========
        conf_mem = prob_mem.max(dim=1).values
        conf_non = prob_non.max(dim=1).values

        entropy_mem = -(prob_mem * torch.log(prob_mem + 1e-8)).sum(dim=1)
        entropy_non = -(prob_non * torch.log(prob_non + 1e-8)).sum(dim=1)

        top2_mem = torch.topk(logit_mem, 2, dim=1).values
        top2_non = torch.topk(logit_non, 2, dim=1).values

        margin_mem = top2_mem[:, 0] - top2_mem[:, 1]
        margin_non = top2_non[:, 0] - top2_non[:, 1]

        def compute_auc(x_mem, x_non):
            x = torch.cat([x_mem, x_non]).cpu().numpy()
            y = np.concatenate([
                np.ones(len(x_mem)),
                np.zeros(len(x_non))
            ])
            try:
                auc = roc_auc_score(y, x)
            except:
                auc = 0.5

            return auc, max(auc, 1 - auc)

        # ========= compute =========
        auc_conf_raw, auc_conf_sym = compute_auc(conf_mem, conf_non)
        auc_ent_raw, auc_ent_sym = compute_auc(entropy_mem, entropy_non)
        auc_mar_raw, auc_mar_sym = compute_auc(margin_mem, margin_non)

        # raw：取最“偏离0.5”的那个（保留方向）
        raw_list = [auc_conf_raw, auc_ent_raw, auc_mar_raw]
        auc_raw = max(raw_list, key=lambda x: abs(x - 0.5))

        # sym：攻击强度
        auc_sym = max(auc_conf_sym, auc_ent_sym, auc_mar_sym)

    return auc_raw



# ==========================================
# 4. Training Worker
# ==========================================
def train_worker(args):
    width, use_dp, epochs, batch_size, clip_norm, noise_mult, gpu_id = args
    device = f'cuda:{gpu_id}' if torch.cuda.is_available() else 'cpu'

    label = f"W:{width} {'DP' if use_dp else 'SGD'}"
    train_loader = DataLoader(TensorDataset(X_train_all, Y_train_all),
                               batch_size=batch_size, shuffle=True)

    X_m, Y_m = X_train_all[:200].to(device), Y_train_all[:200].to(device)
    X_nm, Y_nm = X_test_all[:200].to(device), Y_test_all[:200].to(device)

    model = WideMLP(100, width, 10).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    h = {'ep': [], 'acc': [], 'rank': [], 'mia_auc': []}

    for ep in range(epochs + 1):
        model.train()

        for X_b, Y_b in train_loader:
            X_b, Y_b = X_b.to(device), Y_b.to(device)
            optimizer.zero_grad()

            if not use_dp:
                out, _ = model(X_b)
                loss = criterion(out, Y_b)
                loss.backward()
                optimizer.step()

            else:
                out, _ = model(X_b)
                loss = criterion(out, Y_b)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)

                for p in model.parameters():
                    if p.grad is not None:
                        p.grad += noise_mult * torch.randn_like(p.grad)

                optimizer.step()

        if ep % 50 == 0 or ep == epochs:
            model.eval()

            with torch.no_grad():
                out_te, feat = model(X_nm)
                acc = (out_te.argmax(1) == Y_nm).float().mean().item()

                f_centered = feat - feat.mean(dim=0)
                s = torch.linalg.svdvals(f_centered)
                s = s[s > 1e-5]

                s_norm = s / (s.sum() + 1e-8)
                eff_rank = torch.exp(-torch.sum(s_norm * torch.log(s_norm + 1e-10))).item()

                auc = audit_privacy(model, X_m, Y_m, X_nm, Y_nm, device)

                h['ep'].append(ep)
                h['acc'].append(acc)
                h['rank'].append(eff_rank)
                h['mia_auc'].append(auc)

                print(f"{label} | Ep {ep} | Acc: {acc:.2f} | Rank: {eff_rank:.1f} | MIA-AUC: {auc:.3f}")

    return label, h


# ==========================================
# 5. EXECUTION
# ==========================================
if __name__ == '__main__':
    jobs = [(*cfg, i % NUM_GPUS_AVAILABLE) for i, cfg in enumerate(RUN_CONFIG)]
    results = {}

    with concurrent.futures.ThreadPoolExecutor(max_workers=len(RUN_CONFIG)) as ex:
        for label, h in ex.map(train_worker, jobs):
            results[label] = h


In [ ]:
# --- 1. SETTINGS & CUSTOM LEGEND ---
# Input your labels here in the order they appear in 'results'
CUSTOM_LEGEND_LABELS = ["W50, SGD", "W50, DP-SGD",
                        "W500, SGD", "W500, DP-SGD",
                        "W5000, SGD", "W5000, DP-SGD"
                       ]

plt.rcParams.update({
    'font.size': 18,
    'axes.titlesize': 18,
    'axes.labelsize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14
})

# --- 2. PLOTTING ---
fig, axs = plt.subplots(1, 3, figsize=(22, 7))

# Collect lines for the global legend
lines = []

for label, h in results.items():
    ls = '--' if 'DP' in label else '-'
    lw = 2.0 if 'DP' in label else 2.5
    
    # Plotting each metric
    l1, = axs[0].plot(h['ep'], h['acc'], linestyle=ls, linewidth=lw)
    axs[1].plot(h['ep'], h['rank'], linestyle=ls, linewidth=lw)
    
    # MIA AUC Symmetric Correction
    mia_auc = np.array(h['mia_auc'])
    corrected_auc = np.maximum(mia_auc, 1 - mia_auc)
    axs[2].plot(h['ep'], mia_auc, linestyle=ls, linewidth=lw)
    
    lines.append(l1)

# Titles and labels
titles = ['Test Accuracy', 'Effective Rank', 'MIA AUC (Max)']
for ax, title in zip(axs, titles):
    ax.set_title(title, pad=20)
    ax.set_xlabel("Epochs")
    ax.grid(True, alpha=0.4, linestyle='--')

# --- 3. SPACING & GLOBAL LEGEND ---
# Increase wspace for horizontal breathing room
plt.subplots_adjust(wspace=0.2, bottom=0.25)

fig.legend(
    lines, 
    CUSTOM_LEGEND_LABELS, 
    loc='lower center', 
    bbox_to_anchor=(0.5, 0.02), 
    ncol=3, 
    frameon=True,
    edgecolor='black'
)

# Save line (optional)
plt.savefig('dp2000.png', dpi=300, bbox_inches='tight')

plt.show()